# Initial Set up #

In [212]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
from sklearn.pipeline import Pipeline

from scipy import stats

import xgboost as xgb

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
np.random.seed(42)


In [213]:
train = pd.read_csv('train.csv')
rows, cols = train.shape
print(f"Training Dataset: {rows:,} patient records × {cols} features")

Training Dataset: 80,000 patient records × 40 features


In [214]:
pd.DataFrame({
    "unique_values": train.nunique(),
    "dtype":         train.dtypes,
}).sort_values("unique_values")


,unique_values,dtype
sex,3,object
arrival_season,4,object
shift,4,object
age_group,4,object
insurance_type,5,object
site_id,5,object
mental_status_triage,5,object
triage_acuity,5,int64
arrival_mode,6,object
transport_origin,7,object


# Data Exploration #

In [215]:

# Triage Acuity Distribution
acuity_labels = {
    1: "Level 1\nResuscitation",
    2: "Level 2\nEmergent",
    3: "Level 3\nUrgent",
    4: "Level 4\nLess Urgent",
    5: "Level 5\nNon-Urgent",
}
acuity_colors = ["#D7191C", "#F46D43", "#FDAE61", "#A6D96A", "#1A9641"]

counts = train['triage_acuity'].value_counts().sort_index(ascending=False)
total = counts.sum()
pcts = (counts / total * 100).round(1)

fig = go.Figure()

for level, count in counts.items():
    fig.add_trace(go.Bar(
        x=[acuity_labels[level]],
        y=[count],
        name=acuity_labels[level],
        marker_color=acuity_colors[level - 1],
        marker_line=dict(color="white", width=1.5),
        text=f"<b>{count:,}</b><br>{pcts[level]}%",
        textposition="outside",
        textfont=dict(size=13),
        hovertemplate=(
            f"<b>Triage Level {level}</b><br>"
            f"Count: {count:,}<br>"
            f"Share: {pcts[level]}%<extra></extra>"
        ),
    ))

fig.update_layout(
    title=dict(
        text="<b>Triage Acuity Distribution(Training)</b><br>"
             "<sup>ESI levels 5 (non-urgent) → 1 (most critical)</sup>",
        x=0.5,
        xanchor="center",
        font=dict(size=22, family="Arial"),
    ),
    xaxis=dict(
        title="Triage Acuity Level",
        tickfont=dict(size=12),
        showgrid=False,
    ),
    yaxis=dict(
        title="Number of Patients",
        tickformat=",",
        gridcolor="#e5e5e5",
        showgrid=True,
    ),
    plot_bgcolor="white",
    paper_bgcolor="white",
    showlegend=False,
    bargap=0.25,
    margin=dict(t=110, b=60, l=70, r=40),
    height=520,
    annotations=[
        dict(
            text=f"<i>Total patients: {total:,}</i>",
            xref="paper", yref="paper",
            x=1.0, y=1.07,
            showarrow=False,
            font=dict(size=12, color="gray"),
            xanchor="right",
        )
    ],
)

fig.show()

In [216]:
missing = (train.isna().sum() / len(train) * 100).sort_values(ascending=False)
missing = missing[missing > 0]

print("=" * 45)
print("        Missing Value Proportion")
print("=" * 45)
for col, pct in missing.items():
    count = train[col].isna().sum()
    bar   = "█" * int(pct // 2)
    print(f"  {col:<35} {pct:5.2f}%  ({count:,})")
    print(f"  {bar}")
print("=" * 45)
print(f"  {len(missing)} / {train.shape[1]} columns have missing values")
print("=" * 45)


        Missing Value Proportion
  mean_arterial_pressure               5.18%  (4,146)
  ██
  pulse_pressure                       5.18%  (4,146)
  ██
  systolic_bp                          5.18%  (4,146)
  ██
  shock_index                          5.18%  (4,146)
  ██
  diastolic_bp                         5.18%  (4,146)
  ██
  respiratory_rate                     3.83%  (3,067)
  █
  temperature_c                        0.72%  (574)
  
  7 / 40 columns have missing values


# Pre-processing #

## feature engineering ##

In [217]:
def feature_engineering(df = train):
    column_to_drop = []

    # ---arrival_hour---
    DAYTIME_START = 8
    EVENING_START = 18
    EVENING_END   = 22
    hour = df["arrival_hour"]
    df["is_daytime"] = hour.between(DAYTIME_START, EVENING_START - 1).astype(int)   # 8–17
    df["is_evening"] = hour.between(EVENING_START, EVENING_END).astype(int)          # 18–22
    df["is_night"]   = (~hour.between(DAYTIME_START, EVENING_END)).astype(int)       # 23, 0–7
    column_to_drop.append("arrival_hour")

    # ---age---
    age_bins   = [0, 1, 12, 17, 44, 54, 64, 79, float("inf")]
    age_labels = ["infant", "child", "adolescent", "young_adult",
                "middle_aged_early", "middle_aged_late",
                "senior", "elderly"]
    df["age_group_binned"] = pd.cut(df["age"], bins=age_bins, labels=age_labels, right=True)

    for label in age_labels:
        df[f"is_{label}"] = (df["age_group_binned"] == label).astype(int)
    column_to_drop.append("age")
    column_to_drop.append("age_group_binned")
    column_to_drop.append("age_group")

    # ---triage_nurse_id---
    column_to_drop.append("triage_nurse_id")

    # ---patient_id---(temp: add it for other analysis later)
    column_to_drop.append("patient_id")

    # ---One-hot encoding---
    cols = ['arrival_mode','arrival_season','arrival_month','shift','sex']
    dummies = pd.get_dummies(train[cols], prefix="is", prefix_sep="_", drop_first=False).astype(int)
    df = pd.concat([train, dummies], axis=1)
    print(dummies)
    column_to_drop += cols






    df.drop(columns=column_to_drop, inplace=True)
    return df

train = feature_engineering(train)

       arrival_month  is_ambulance  is_brought_by_family  is_helicopter  \
0                  5             0                     0              0   
1                  4             0                     0              0   
2                  4             0                     0              0   
3                  3             0                     0              0   
4                  5             0                     0              0   
...              ...           ...                   ...            ...   
79995              9             0                     0              0   
79996             11             0                     0              0   
79997              7             1                     0              0   
79998             10             0                     1              0   
79999             11             0                     0              0   

       is_police  is_transfer  is_walk-in  is_autumn  is_spring  is_summer  \
0              0     

In [218]:
train

,site_id,arrival_day,language,insurance_type,transport_origin,pain_location,mental_status_triage,chief_complaint_system,num_prior_ed_visits_12m,num_prior_admissions_12m,num_active_medications,num_comorbidities,systolic_bp,diastolic_bp,mean_arterial_pressure,pulse_pressure,heart_rate,respiratory_rate,temperature_c,spo2,gcs_total,pain_score,weight_kg,height_cm,bmi,shock_index,news2_score,disposition,ed_los_hours,triage_acuity,is_daytime,is_evening,is_night,is_infant,is_child,is_adolescent,is_young_adult,is_middle_aged_early,is_middle_aged_late,is_senior,is_elderly,is_ambulance,is_brought_by_family,is_helicopter,is_police,is_transfer,is_walk-in,is_autumn,is_spring,is_summer,is_winter,is_afternoon,is_evening,is_morning,is_night,is_F,is_M,is_Other
0,SITE-TMP-01,Monday,Finnish,public,public_space,extremity,drowsy,neurological,0,0,4,8,79.0,57.5,64.7,21.5,57.3,17.9,37.0,92.1,14,7,52.3,165.4,19.1,0.725,8,discharged,7.35,2,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,1,0,0,1,0
1,SITE-HEL-01,Thursday,Russian,military,home,extremity,alert,genitourinary,0,0,10,8,131.7,93.4,106.2,38.3,97.3,17.2,36.9,99.4,15,-1,73.3,164.4,27.1,0.739,1,discharged,0.70,5,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,1,0,0,0,0,1,0,1,0,0
2,SITE-HEL-02,Saturday,English,none,nursing_home,abdomen,alert,other,0,0,13,14,94.7,83.3,87.1,11.4,75.6,14.7,37.3,100.0,15,3,77.1,183.7,22.8,0.798,2,discharged,0.63,5,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,1,0,0,0,0,1,0,0,1,0
3,SITE-HEL-02,Sunday,Finnish,private,outdoor,abdomen,alert,dermatological,3,1,4,3,134.2,51.8,79.3,82.4,109.0,17.6,38.2,96.0,15,7,49.6,172.6,16.6,0.812,2,discharged,1.99,3,0,0,1,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0,1,0,0
4,SITE-HEL-02,Tuesday,Finnish,public,home,multiple,alert,dermatological,2,0,10,17,140.1,75.4,97.0,64.7,113.7,17.6,36.6,99.1,15,4,71.9,173.4,23.9,0.812,2,transferred,3.58,3,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,1,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
79995,SITE-HEL-02,Wednesday,Russian,none,home,extremity,confused,musculoskeletal,1,0,6,6,126.1,83.7,97.8,42.4,89.6,17.1,36.9,96.4,15,6,53.3,166.9,19.1,0.711,0,discharged,2.86,4,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,1,0,1,0,0
79996,SITE-OUL-01,Tuesday,Other,public,outdoor,none,alert,other,1,0,7,6,141.9,87.2,105.4,54.7,62.7,18.6,37.6,96.0,15,9,89.6,175.1,29.2,0.442,0,lama,4.61,3,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,1,0,0,1,0
79997,SITE-HEL-02,Thursday,Estonian,private,public_space,chest,alert,genitourinary,0,0,5,2,136.9,80.2,99.1,56.7,75.4,11.7,36.8,99.6,15,0,75.5,171.7,25.6,0.551,0,discharged,1.40,5,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,1,0,1,0
79998,SITE-OUL-01,Friday,Finnish,public,public_space,back,drowsy,ophthalmic,4,3,8,9,43.1,45.6,44.8,-2.5,109.4,30.6,39.7,70.5,6,10,67.6,166.9,24.3,2.538,15,admitted,6.45,1,1,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0,0,1,0,0,0,1,0,0
